In [1]:
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

#### Select the model to use

In [3]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

#### Configure quantization

In [16]:
def check_cuda():
    if torch.cuda.is_available():
        print(f"✅ CUDA is available. Found {torch.cuda.device_count()} device(s).")
        print(f"Device Name: {torch.cuda.get_device_name(0)}")
        print("Will use quantizaiton")
        return True
    else:
        print("❌ CUDA is NOT available. Skipping quantization")
        return False

USE_QUANTIZATION = check_cuda()

❌ CUDA is NOT available. Skipping quantization


In [17]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

In [18]:
if USE_QUANTIZATION:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        use_cache=False
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,  # Native half-precision for Mac
    use_cache=False
)

`torch_dtype` is deprecated! Use `dtype` instead!


#### Load the tokenizer

In [19]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# Qwen has a real pad token (<|endoftext|> usually).
# We just ensure padding is on the right for the Trainer.
tokenizer.padding_side = "right"

#### Prepare for training

In [20]:
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_BIAS = "none"

In [21]:
if USE_QUANTIZATION:
    model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


#### Tokenize the dataset

In [22]:
# Load a raw dataset
dataset = load_dataset("mlabonne/guanaco-llama2-1k", split="train")

# Define the max context length 
MAX_LENGTH = 1024 

In [23]:
def format_and_tokenize(example):
    # Qwen works best with ChatML format, but for simple finetuning, 
    # we can just feed it text.
    text = example['text'] 
    
    # Tokenize
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH, # 1.5B model can handle long context, but 1024 is safe for memory
        padding="max_length",
    )
    
    input_ids = tokenized["input_ids"]
    labels = input_ids[:]
    
    # Mask padding tokens in labels so the model doesn't train on them
    # (Qwen uses specific pad token IDs, usually 151643 or similar)
    pad_token_id = tokenizer.pad_token_id
    for i, token_id in enumerate(labels):
        if token_id == pad_token_id:
            labels[i] = -100
            
    tokenized["labels"] = labels
    return tokenized

tokenized_dataset = dataset.map(format_and_tokenize)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

#### Data collector

In [24]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

#### Training

In [25]:
OUTPUT_DIR = "./qwen_1.5b_finetune"
LEARNING_RATE = 2e-4
LOGGING_STEPS = 10
MAX_STEPS = 100
TRAIN_BATCH_SIZE = 1 # Lowest memory footprint
GRADIENT_ACCUMULATION_STEPS = 4

In [26]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=TRAIN_BATCH_SIZE, 
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    fp16=True,
    logging_steps=LOGGING_STEPS,
    max_steps=MAX_STEPS,
    save_strategy="no",
    report_to="none"
)

In [27]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

In [ ]:
trainer.train()

/Users/wojciechdrozdz/uni/Magisterka/Semestr III/Praca Magisterska/lora-fine-tuning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


In [ ]:
# 8. Inference Test (See if it worked)
text = "Human: What is the capital of France? Assistant:"
inputs = tokenizer(text, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))